#  ⭐ `fat_medicamentos`


In [1]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, normalize_date_column, normalize_rows, apply_mappings, fuzzy_merge##, normalize_duration, expandir_gravidade_wide
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [2]:
path = Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO'],
      dtype='object')

In [3]:
bronze.head(2)

,IDENTIFICACAO_NOTIFICACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,CODIGO_ATC,DETENTOR_REGISTRO,CONCENTRACAO,COMPONENTE_SUSPEITO,ACAO_ADOTADA,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,INDICACAO_MEDDRA,INDICACAO_RELATADA_NOTIFICADOR_INICIAL,DOSE,FREQUENCIA_DOSE,POSOLOGIA,DURACAO,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_MAE_PAI,NUMELO_LOTE,ATUALIZADO
0,BR-ANVISA-300000004,Suspeito,Oxacilina,Oxacillin sodium,J01CF,Teuto,None,None,None,None,None,None,250 milligram (mg),6 hora,None,4 dia,20181124,None,None,infusão intravenosa,None,5833018,2025-11-17
1,BR-ANVISA-300000005,Suspeito,Paracemol,Paracetamol,N02BE,None,None,None,Retirada do medicamento,None,None,None,None,None,None,None,20181122,20181122,None,oral,None,None,2025-11-17


# 🥈 Silver

normalized data, medium quality


In [4]:
silver = bronze.copy()
silver.shape

(552887, 23)

## ✅ hash

In [5]:
from utils import build_row_hash

hash_cols = ['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE']  # as colunas que devem entrar no hash
silver["HASH_BRONZE"] = build_row_hash(silver, hash_cols, algo="sha256", sep="|")

## ✅ Missing

In [6]:
silver.isna().sum()

IDENTIFICACAO_NOTIFICACAO                            0
RELACAO_MEDICAMENTO_EVENTO                        1000
NOME_MEDICAMENTO_WHODRUG                          5159
PRINCIPIOS_ATIVOS_WHODRUG                         5649
CODIGO_ATC                                        5159
DETENTOR_REGISTRO                               426201
CONCENTRACAO                                    460259
COMPONENTE_SUSPEITO                             496737
ACAO_ADOTADA                                    267347
PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO    534294
INDICACAO_MEDDRA                                325534
INDICACAO_RELATADA_NOTIFICADOR_INICIAL          315982
DOSE                                            339554
FREQUENCIA_DOSE                                   3583
POSOLOGIA                                       275637
DURACAO                                         473049
INICIO_ADMINISTRACAO                            288439
FIM_ADMINISTRACAO                               395839
FORMA_FARM

In [7]:
hist_silver = silver.copy()
for col in silver.select_dtypes(include=['object']).columns:
    silver[col] = normalize_rows(silver[col])

In [8]:
hist_silver.isna().sum()

IDENTIFICACAO_NOTIFICACAO                            0
RELACAO_MEDICAMENTO_EVENTO                        1000
NOME_MEDICAMENTO_WHODRUG                          5159
PRINCIPIOS_ATIVOS_WHODRUG                         5649
CODIGO_ATC                                        5159
DETENTOR_REGISTRO                               426201
CONCENTRACAO                                    460259
COMPONENTE_SUSPEITO                             496737
ACAO_ADOTADA                                    267347
PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO    534294
INDICACAO_MEDDRA                                325534
INDICACAO_RELATADA_NOTIFICADOR_INICIAL          315982
DOSE                                            339554
FREQUENCIA_DOSE                                   3583
POSOLOGIA                                       275637
DURACAO                                         473049
INICIO_ADMINISTRACAO                            288439
FIM_ADMINISTRACAO                               395839
FORMA_FARM

In [9]:
hist_silver.shape

(552887, 24)

## ✅ DATAS

In [10]:
colunas_data = ["INICIO_ADMINISTRACAO", "FIM_ADMINISTRACAO"]

In [11]:
hist_silver[colunas_data].info()
hist_silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 2 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   INICIO_ADMINISTRACAO  264448 non-null  object
 1   FIM_ADMINISTRACAO     157048 non-null  object
dtypes: object(2)
memory usage: 8.4+ MB


,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO
0,20181124,None
1,20181122,20181122
2,20181103,20181115
3,20181016,20181115
4,20181024,None


In [12]:
for col in colunas_data:
    if col in hist_silver.columns:
        hist_silver[col] = normalize_date_column(hist_silver[col])

In [13]:
hist_silver[colunas_data].info()
hist_silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 2 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   INICIO_ADMINISTRACAO  220969 non-null  datetime64[ns]
 1   FIM_ADMINISTRACAO     142767 non-null  datetime64[ns]
dtypes: datetime64[ns](2)
memory usage: 8.4 MB


,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO
0,2018-11-24,NaT
1,2018-11-22,2018-11-22
2,2018-11-03,2018-11-15
3,2018-10-16,2018-11-15
4,2018-10-24,NaT


In [14]:
hist_silver.shape

(552887, 24)

## ✅ Mappings

- RELACAO_MED_EVENTO
- COMPONENTE_SUSPEITO
- ACAO_ADOTADA

In [15]:
# RELACAO_MEDICAMENTO_EVENTO
relacao_medicamento_evento_map = {
    "SUSPEITO": 1,
    "CONCOMITANTE": 2,
    "MEDICAMENTO NAO ADMINISTRADO": 3,
    "INTERACAO": 4,
}
# COMPONENTE_SUSPEITO
componente_suspeito_map = {
    "PRINCIPIO ATIVO": 1, "EXCIPIENTE, NAO CLASSIFICADO": 2, "SOLVENTE": 3, "CORANTE": 4,
    "CONSERVANTE": 5, "AGENTE FLAVORIZADOR": 6, "EXCESSO PERCENTUAL": 7, "ANTIOXIDANTE": 8,
    "ESTABILIZANTE": 9
}

# ACAO_ADOTADA
acao_adotada_map = {
    "RETIRADA DO MEDICAMENTO": 1, "SEM ALTERACAO DA DOSE": 2, "NAO APLICAVEL": 3,
    "REDUCAO DA DOSE": 4, "AUMENTO DA DOSE": 5
}
mappings_config = [
    {
        "kind": "dict",
        "col": "RELACAO_MEDICAMENTO_EVENTO",
        "map": relacao_medicamento_evento_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": True,
    },
    {
        "kind": "dict",
        "col": "COMPONENTE_SUSPEITO",
        "map": componente_suspeito_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": True,
    },
    {
        "kind": "dict",
        "col": "ACAO_ADOTADA",
        "map": acao_adotada_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": True,
    }]

In [16]:
hist_silver = apply_mappings(hist_silver, mappings_config)

In [17]:
hist_silver.shape

(552887, 27)

In [18]:
hist_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 27 columns):
 #   Column                                        Non-Null Count   Dtype         
---  ------                                        --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                     552887 non-null  object        
 1   NOME_MEDICAMENTO_WHODRUG                      547728 non-null  object        
 2   PRINCIPIOS_ATIVOS_WHODRUG                     547238 non-null  object        
 3   CODIGO_ATC                                    547728 non-null  object        
 4   DETENTOR_REGISTRO                             126686 non-null  object        
 5   CONCENTRACAO                                  92628 non-null   object        
 6   PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO  18593 non-null   object        
 7   INDICACAO_MEDDRA                              227353 non-null  object        
 8   INDICACAO_RELATADA_NOTIFICADOR_INICIAL        236905 n

## ✅ DURACAO

In [19]:
from utils import normalize_duracao

In [20]:
hist_silver = normalize_duracao(hist_silver, coluna="DURACAO")

In [21]:
hist_silver[['DURACAO_TIPO_CHAVE','DURACAO_TIPO_VALOR','DURACAO_VALOR']].head()

,DURACAO_TIPO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_VALOR
0,dia,4,4.0
1,desconhecido,0,NaN
2,desconhecido,0,NaN
3,desconhecido,0,NaN
4,desconhecido,0,NaN


In [22]:
hist_silver.shape

(552887, 30)

In [23]:
hist_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 30 columns):
 #   Column                                        Non-Null Count   Dtype         
---  ------                                        --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                     552887 non-null  object        
 1   NOME_MEDICAMENTO_WHODRUG                      547728 non-null  object        
 2   PRINCIPIOS_ATIVOS_WHODRUG                     547238 non-null  object        
 3   CODIGO_ATC                                    547728 non-null  object        
 4   DETENTOR_REGISTRO                             126686 non-null  object        
 5   CONCENTRACAO                                  92628 non-null   object        
 6   PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO  18593 non-null   object        
 7   INDICACAO_MEDDRA                              227353 non-null  object        
 8   INDICACAO_RELATADA_NOTIFICADOR_INICIAL        236905 n

## ✅ VIA_ADM

In [24]:
from utils.med_normalize_via_adm import normalizar_via_fuzzy


In [25]:
hist_silver["VIA_ADMINISTRACAO_CHAVE"] = (
    hist_silver["VIA_ADMINISTRACAO"]
    .astype(str)
    .apply(lambda txt: normalizar_via_fuzzy(
        txt,
        score_threshold=80,
        return_numeric=True,
    ))
)

hist_silver["VIA_ADMINISTRACAO_MAE_PAI_CHAVE"] = (
    hist_silver["VIA_ADMINISTRACAO_MAE_PAI"]
    .astype(str)
    .apply(lambda txt: normalizar_via_fuzzy(
        txt,
        score_threshold=80,
        return_numeric=True,
    ))
)


In [26]:
hist_silver["VIA_ADMINISTRACAO_CHAVE"].value_counts(dropna=False)

VIA_ADMINISTRACAO_CHAVE
5     360787
6     144719
0      41956
2       1927
9       1417
3        672
1        554
4        323
8        286
10       161
7         85
Name: count, dtype: int64

In [27]:
hist_silver["VIA_ADMINISTRACAO_MAE_PAI_CHAVE"].value_counts(dropna=False)

VIA_ADMINISTRACAO_MAE_PAI_CHAVE
5    552778
6        78
0        27
1         2
2         2
Name: count, dtype: int64

In [28]:
hist_silver.shape

(552887, 32)

## ✅ CONCENTRACAO

In [29]:
from utils.med_normalize_concentracao import normalize_concentracao


In [30]:
hist_silver = normalize_concentracao(hist_silver, col="CONCENTRACAO")

In [31]:
hist_silver.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'NOME_MEDICAMENTO_WHODRUG',
       'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC', 'DETENTOR_REGISTRO',
       'CONCENTRACAO', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO', 'HASH_BRONZE',
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       'COMPONENTE_SUSPEITO_VALOR', 'COMPONENTE_SUSPEITO_CHAVE',
       'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE', 'DURACAO_TIPO_CHAVE',
       'DURACAO_TIPO_VALOR', 'DURACAO_VALOR', 'VIA_ADMINISTRACAO_CHAVE',
       'VIA_ADMINISTRACAO_MAE_PAI_CHAVE', 'CONCENTRACAO_TIPO_CHAVE',
       'CONCENTRACAO_TIPO_VALOR', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR'],
   

In [32]:
hist_silver.query("CONCENTRACAO_TIPO_CHAVE!='desconhecido'")[['CONCENTRACAO','CONCENTRACAO_TIPO_CHAVE',
       'CONCENTRACAO_TIPO_VALOR', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR']].head()

,CONCENTRACAO,CONCENTRACAO_TIPO_CHAVE,CONCENTRACAO_TIPO_VALOR,CONCENTRACAO_VALOR,CONCENTRACAO_VALOR_NUMERADOR,CONCENTRACAO_VALOR_DENOMINADOR
928,2000mg/500mg,milligram per milligram (mg/mg),7,4.0,2000.0,500.0
1046,250 mg,milligram (mg),1,250.0,250.0,NaN
1182,150 mg,milligram (mg),1,150.0,150.0,NaN
1183,25 mg,milligram (mg),1,25.0,25.0,NaN
1219,2mg/mL,milligram per millilitre (mg/mL),6,2.0,2.0,1.0


In [33]:
hist_silver.CONCENTRACAO_TIPO_VALOR.value_counts(dropna=False)

CONCENTRACAO_TIPO_VALOR
0    467578
1     43375
6     29106
2      6598
5      2449
4      2058
3       688
9       626
7       351
8        58
Name: count, dtype: Int64

In [34]:
hist_silver.CONCENTRACAO_TIPO_CHAVE.value_counts(dropna=False)

CONCENTRACAO_TIPO_CHAVE
desconhecido                        467578
milligram (mg)                       43375
milligram per millilitre (mg/mL)     29106
gram (g)                              6598
percent (%)                           2449
millilitre (mL)                       2058
microgram (ug)                         688
gram per millilitre (g/mL)             626
milligram per milligram (mg/mg)        351
milligram per gram (mg/g)               58
Name: count, dtype: int64

## ✅ DOSE

In [35]:
from utils.med_normalize_dose import normalize_dose

In [36]:
hist_silver = normalize_dose(hist_silver, col="DOSE")


In [37]:
hist_silver.query("DOSE_TIPO_CHAVE!='desconhecido'")[['DOSE','DOSE_TIPO_CHAVE', 'DOSE_TIPO_VALOR', 'DOSE_VALOR']].head()

,DOSE,DOSE_TIPO_CHAVE,DOSE_TIPO_VALOR,DOSE_VALOR
0,250 milligram (mg),milligram (mg),1,250.0
2,250 milligram (mg),milligram (mg),1,250.0
7,1200000 international unit ([iU]),international unit ([iU]),8,1200000.0
10,183 milligram (mg),milligram (mg),1,183.0
12,2 milligram (mg),milligram (mg),1,2.0


In [38]:
hist_silver.shape

(552887, 40)

## Fuzzy

### ✅ DETENTOR_REGISTRO

In [39]:
hist_dim_detentor_registro = pd.read_parquet(Path(project_root) / "data/02_silver/hist_dim_detentor_registro/hist_dim_detentor_registro.parquet")
hist_dim_detentor_registro[['DETENTOR_REGISTRO','DETENTOR_REGISTRO_CHAVE', 'DETENTOR_REGISTRO_VALOR']].drop_duplicates().head()

,DETENTOR_REGISTRO,DETENTOR_REGISTRO_CHAVE,DETENTOR_REGISTRO_VALOR
0,ABACUS MEDICINE,1,ABACUS MEDICINE
1,ABBOLT,2,ABBOT
2,ABBOT,2,ABBOT
3,ABBOT BRASIL,2,ABBOT
4,ABBOT L: 0071/20 V: 31/01/2022,2,ABBOT


In [40]:
hist_silver = fuzzy_merge(
    dim_df = hist_dim_detentor_registro,
    fact_df = hist_silver,
    dim_id_col="DETENTOR_REGISTRO_CHAVE",
    dim_text_col="DETENTOR_REGISTRO",
    fact_text_col="DETENTOR_REGISTRO",
    threshold=95, 
    suffix="" , 
    dedupe_on=False,
)

In [41]:
hist_silver['DETENTOR_REGISTRO_CHAVE'] = hist_silver['DETENTOR_REGISTRO_CHAVE'].fillna(0) 
hist_silver['DETENTOR_REGISTRO_CHAVE'].value_counts(dropna=False).head()

DETENTOR_REGISTRO_CHAVE
0      448886
223     10243
237      9045
83       7214
161      5404
Name: count, dtype: Int64

In [42]:
silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 24 columns):
 #   Column                                        Non-Null Count   Dtype 
---  ------                                        --------------   ----- 
 0   IDENTIFICACAO_NOTIFICACAO                     552887 non-null  object
 1   RELACAO_MEDICAMENTO_EVENTO                    551887 non-null  object
 2   NOME_MEDICAMENTO_WHODRUG                      547728 non-null  object
 3   PRINCIPIOS_ATIVOS_WHODRUG                     547238 non-null  object
 4   CODIGO_ATC                                    547728 non-null  object
 5   DETENTOR_REGISTRO                             126686 non-null  object
 6   CONCENTRACAO                                  92628 non-null   object
 7   COMPONENTE_SUSPEITO                           56150 non-null   object
 8   ACAO_ADOTADA                                  285540 non-null  object
 9   PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO  18593 non-nul

### ✅ FORMA_FARMACEUTICA

In [43]:
hist_dim_forma_farmaceutica = pd.read_parquet(Path(project_root) / "data/02_silver/hist_dim_forma_farmaceutica/hist_dim_forma_farmaceutica.parquet")
hist_dim_forma_farmaceutica[['FORMA_FARMACEUTICA','FORMA_FARMACEUTICA_CHAVE', 'FORMA_FARMACEUTICA_VALOR']].drop_duplicates().head()

,FORMA_FARMACEUTICA,FORMA_FARMACEUTICA_CHAVE,FORMA_FARMACEUTICA_VALOR
0,Adesivo Transdérmico,1,Adesivo transdermico
1,adesivo,1,Adesivo transdermico
2,adesivo transdérmico,1,Adesivo transdermico
3,Adesivo,1,Adesivo transdermico
5,ADESIVO,1,Adesivo transdermico


In [44]:
hist_silver.shape

(552887, 42)

In [45]:
hist_silver = fuzzy_merge(
    dim_df = hist_dim_forma_farmaceutica,
    fact_df = hist_silver,
    dim_id_col="FORMA_FARMACEUTICA_CHAVE",
    dim_text_col="FORMA_FARMACEUTICA",
    fact_text_col="FORMA_FARMACEUTICA",
    threshold=95, 
    suffix="" , 
    dedupe_on=False,
)

In [46]:
hist_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 44 columns):
 #   Column                                        Non-Null Count   Dtype         
---  ------                                        --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                     552887 non-null  object        
 1   NOME_MEDICAMENTO_WHODRUG                      547728 non-null  object        
 2   PRINCIPIOS_ATIVOS_WHODRUG                     547238 non-null  object        
 3   CODIGO_ATC                                    547728 non-null  object        
 4   DETENTOR_REGISTRO                             104001 non-null  object        
 5   CONCENTRACAO                                  92628 non-null   object        
 6   PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO  18593 non-null   object        
 7   INDICACAO_MEDDRA                              227353 non-null  object        
 8   INDICACAO_RELATADA_NOTIFICADOR_INICIAL        236905 n

In [47]:
hist_silver['FORMA_FARMACEUTICA_CHAVE'] = hist_silver['FORMA_FARMACEUTICA_CHAVE'].fillna(0) 

In [48]:
hist_silver['FORMA_FARMACEUTICA_CHAVE'].value_counts(dropna=False)

FORMA_FARMACEUTICA_CHAVE
0     405151
14     64952
4      34881
12     14933
16     12701
11     10297
2       5036
15      1394
10      1336
3        572
1        324
20       237
8        220
19       157
18       124
9        120
13       116
6        112
5         89
7         82
17        53
Name: count, dtype: Int64

### ✅ INDICACAO_MEDDRA

In [49]:
dim_soc_llt = pd.read_parquet(Path(project_root) / "data/03_gold/dim_soc_llt/dim_soc_llt.parquet")
dim_soc_llt["PK_LLT"] = dim_soc_llt["PK_LLT"].astype(int) 
dim_soc_llt[['PK_LLT','REACAO_EVTO_ADVERSO_MEDDRA_LLT']]
dim_soc_llt.head()

,PK_SOC,SOC,HLGT,HLT,PT,PK_LLT,REACAO_EVTO_ADVERSO_MEDDRA_LLT
0,26,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,1,Amamentação
1,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,2,Abstinência sexual
2,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,3,Atividade sexual aumentada
3,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,4,Não consumação
4,26,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,5,Neonato


In [50]:
hist_silver.INDICACAO_MEDDRA.value_counts().head()

INDICACAO_MEDDRA
USO DE MEDICAMENTO PARA INDICAÇÃO DESCONHECIDA    19595
PRODUTO USADO PARA INDICAÇÃO DESCONHECIDA         16515
DOENÇA DE CROHN                                    6875
ARTRITE REUMATOIDE                                 6207
HIPERTENSÃO                                        4399
Name: count, dtype: int64

In [51]:
hist_silver = fuzzy_merge(
    dim_df=dim_soc_llt[['PK_LLT', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT']],
    fact_df=hist_silver,
    dim_id_col="PK_LLT",
    dim_text_col="REACAO_EVTO_ADVERSO_MEDDRA_LLT",
    fact_text_col="INDICACAO_MEDDRA",
    threshold=98,
    suffix="_LK"  # opcional; default "_MATCH"
)
# hist_silver terá PK_LLT_LK (Int64), INDICACAO_MEDDRA_LK, INDICACAO_MEDDRA_SCORE
#hist_silver[['PK_LLT_INDICACAO_MEDDRA']] = hist_silver[['PK_LLT_MATCH']]

### ✅ INDICACAO_RELATADA_NOTIFICADOR_INICIAL

In [52]:
hist_silver.INDICACAO_RELATADA_NOTIFICADOR_INICIAL.value_counts().head()

INDICACAO_RELATADA_NOTIFICADOR_INICIAL
PRODUCT USED FOR UNKNOWN INDICATION    11420
DRUG USE FOR UNKNOWN INDICATION         6094
DOENÇA DE CROHN                         3180
DIABETES                                3010
HIPERTENSÃO                             2838
Name: count, dtype: int64

In [53]:
hist_silver = fuzzy_merge(
    dim_df = dim_soc_llt,
    fact_df = hist_silver,
    dim_id_col="PK_LLT",
    dim_text_col="REACAO_EVTO_ADVERSO_MEDDRA_LLT",
    fact_text_col="INDICACAO_RELATADA_NOTIFICADOR_INICIAL",
    threshold = 98)

#hist_silver[['PK_LLT_INDICACAO_RELATADA_NOTIFICADOR_INICIAL']] = hist_silver[['PK_LLT_MATCH']]

## Pivot

### ✅ PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO

In [54]:
from utils import expandir_lista_wide

In [55]:
hist_silver = expandir_lista_wide(
    hist_silver,
    col="PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO",
    prefix="PROB_ADIC_",   # se quiser um prefixo mais curto/bonito
)


In [56]:

hist_silver.filter(like="PROB_ADIC_").head(2)

,PROB_ADIC_ABUSO,PROB_ADIC_ERRO_DE_MEDICACAO,PROB_ADIC_EXPOSICAO_OCUPACIONAL,PROB_ADIC_FALSIFICACAO,PROB_ADIC_LOTES_TESTADOS_E_DENTRO_DAS_ESPECIFICACOES,PROB_ADIC_LOTES_TESTADOS_E_FORA_DAS_ESPECIFICACOES,PROB_ADIC_MEDICAMENTO_TOMADO_FORA_DA_DATA_DE_VALIDADE,PROB_ADIC_MEDICAMENTO_TOMADO_PELO_PAI,PROB_ADIC_SUPERDOSE,PROB_ADIC_USO_INCORRETO,PROB_ADIC_USO_OFF_LABEL_USO_SEM_REGISTRO
0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0


In [57]:
hist_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 61 columns):
 #   Column                                                 Non-Null Count   Dtype         
---  ------                                                 --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                              552887 non-null  object        
 1   NOME_MEDICAMENTO_WHODRUG                               547728 non-null  object        
 2   PRINCIPIOS_ATIVOS_WHODRUG                              547238 non-null  object        
 3   CODIGO_ATC                                             547728 non-null  object        
 4   DETENTOR_REGISTRO                                      104001 non-null  object        
 5   CONCENTRACAO                                           92628 non-null   object        
 6   PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO           552887 non-null  object        
 7   INDICACAO_MEDDRA                                       2

### ✅ CODIGO_ATC

In [58]:
from utils import desagrupar_colunas_pipe

In [59]:
colunas_para_desagrupar = ['ATC_CODE_LEVEL_4']
print(f"Shape antes da desagregação: {hist_silver.shape}") 
hist_silver_dedup = desagrupar_colunas_pipe(hist_silver.rename(columns={'CODIGO_ATC': 'ATC_CODE_LEVEL_4'}), colunas_para_desagrupar)
print(f"Shape depois da desagregação: {hist_silver_dedup.shape}") 

Shape antes da desagregação: (552887, 61)
Shape depois da desagregação: (748297, 61)


In [61]:
hist_silver_dedup.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'NOME_MEDICAMENTO_WHODRUG',
       'PRINCIPIOS_ATIVOS_WHODRUG', 'ATC_CODE_LEVEL_4', 'DETENTOR_REGISTRO',
       'CONCENTRACAO', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO', 'HASH_BRONZE',
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       'COMPONENTE_SUSPEITO_VALOR', 'COMPONENTE_SUSPEITO_CHAVE',
       'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE', 'DURACAO_TIPO_CHAVE',
       'DURACAO_TIPO_VALOR', 'DURACAO_VALOR', 'VIA_ADMINISTRACAO_CHAVE',
       'VIA_ADMINISTRACAO_MAE_PAI_CHAVE', 'CONCENTRACAO_TIPO_CHAVE',
       'CONCENTRACAO_TIPO_VALOR', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR'

In [ ]:
===

In [62]:


hash_cols_silver = ['IDENTIFICACAO_NOTIFICACAO', 
       'NOME_MEDICAMENTO_WHODRUG',
       'PRINCIPIOS_ATIVOS_WHODRUG', 
       'ATC_CODE_LEVEL_4', 
       
       'DETENTOR_REGISTRO',
       'DETENTOR_REGISTRO_SCORE', 
       'DETENTOR_REGISTRO_CHAVE',
       
       
       'CONCENTRACAO',
       'CONCENTRACAO_TIPO_CHAVE',
       'CONCENTRACAO_TIPO_VALOR', 
       'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 
       'CONCENTRACAO_VALOR_DENOMINADOR',

       'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'PROB_ADIC_ERRO_DE_MEDICACAO', 'PROB_ADIC_EXPOSICAO_OCUPACIONAL',
       'PROB_ADIC_FALSIFICACAO',
       'PROB_ADIC_LOTES_TESTADOS_E_DENTRO_DAS_ESPECIFICACOES',
       'PROB_ADIC_LOTES_TESTADOS_E_FORA_DAS_ESPECIFICACOES',
       'PROB_ADIC_MEDICAMENTO_TOMADO_FORA_DA_DATA_DE_VALIDADE',
       'PROB_ADIC_MEDICAMENTO_TOMADO_PELO_PAI', 'PROB_ADIC_SUPERDOSE',
       'PROB_ADIC_USO_INCORRETO', 'PROB_ADIC_USO_OFF_LABEL_USO_SEM_REGISTRO','PROB_ADIC_ABUSO',

       'INDICACAO_MEDDRA', 
       'INDICACAO_MEDDRA_SCORE', 
       'PK_LLT_LK', 
       'INDICACAO_MEDDRA_LK',
       
       'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 
       'INDICACAO_RELATADA_NOTIFICADOR_INICIAL_SCORE', 
       'PK_LLT_MATCH',
       'INDICACAO_RELATADA_NOTIFICADOR_INICIAL_MATCH',
       
       
       'DOSE',
       'DOSE_TIPO_CHAVE', 
       'DOSE_TIPO_VALOR', 
       'DOSE_VALOR',


       'FREQUENCIA_DOSE', 
       'POSOLOGIA', 
       
       'DURACAO',
       'DURACAO_TIPO_CHAVE',
       'DURACAO_TIPO_VALOR', 
       'DURACAO_VALOR',

       'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 

       'FORMA_FARMACEUTICA', 
       'FORMA_FARMACEUTICA_SCORE', 
       'FORMA_FARMACEUTICA_CHAVE', 
       
       
       'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_CHAVE',

       'VIA_ADMINISTRACAO_MAE_PAI',
       'VIA_ADMINISTRACAO_MAE_PAI_CHAVE', 
       
       'NUMELO_LOTE', 
       #'ATUALIZADO', 
       'HASH_BRONZE',
       
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 
       'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       
       'COMPONENTE_SUSPEITO_VALOR', 
       'COMPONENTE_SUSPEITO_CHAVE',
       
       'ACAO_ADOTADA_VALOR', 
       'ACAO_ADOTADA_CHAVE', 
       
       
       
       ]
       
hist_silver_dedup["HASH_SILVER"] = build_row_hash(hist_silver_dedup, hash_cols_silver, algo="sha256", sep="|")

In [63]:
hist_silver_dedup.to_parquet(Path(project_root) / "data/02_silver/hist_fat_medicamentos/hist_fat_medicamentos.parquet", index=False)

# 🥇 Gold

In [64]:
hist_silver_dedup.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'NOME_MEDICAMENTO_WHODRUG',
       'PRINCIPIOS_ATIVOS_WHODRUG', 'ATC_CODE_LEVEL_4', 'DETENTOR_REGISTRO',
       'CONCENTRACAO', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO', 'HASH_BRONZE',
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       'COMPONENTE_SUSPEITO_VALOR', 'COMPONENTE_SUSPEITO_CHAVE',
       'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE', 'DURACAO_TIPO_CHAVE',
       'DURACAO_TIPO_VALOR', 'DURACAO_VALOR', 'VIA_ADMINISTRACAO_CHAVE',
       'VIA_ADMINISTRACAO_MAE_PAI_CHAVE', 'CONCENTRACAO_TIPO_CHAVE',
       'CONCENTRACAO_TIPO_VALOR', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR'

In [66]:
# Lista base de colunas (use sempre os nomes de origem aqui)
gold_cols = [
    'IDENTIFICACAO_NOTIFICACAO',
    'NOME_MEDICAMENTO_WHODRUG',
    'PRINCIPIOS_ATIVOS_WHODRUG',
    'ATC_CODE_LEVEL_4',
    # 'DETENTOR_REGISTRO',
    'DETENTOR_REGISTRO_SCORE',
    'DETENTOR_REGISTRO_CHAVE',

    # 'CONCENTRACAO',
    'CONCENTRACAO_TIPO_CHAVE',
    'CONCENTRACAO_TIPO_VALOR',
    'CONCENTRACAO_VALOR',
    'CONCENTRACAO_VALOR_NUMERADOR',
    'CONCENTRACAO_VALOR_DENOMINADOR',

    # 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
    'PROB_ADIC_ERRO_DE_MEDICACAO',
    'PROB_ADIC_EXPOSICAO_OCUPACIONAL',
    'PROB_ADIC_FALSIFICACAO',
    'PROB_ADIC_LOTES_TESTADOS_E_DENTRO_DAS_ESPECIFICACOES',
    'PROB_ADIC_LOTES_TESTADOS_E_FORA_DAS_ESPECIFICACOES',
    'PROB_ADIC_MEDICAMENTO_TOMADO_FORA_DA_DATA_DE_VALIDADE',
    'PROB_ADIC_MEDICAMENTO_TOMADO_PELO_PAI',
    'PROB_ADIC_SUPERDOSE',
    'PROB_ADIC_USO_INCORRETO',
    'PROB_ADIC_USO_OFF_LABEL_USO_SEM_REGISTRO',
    'PROB_ADIC_ABUSO',

    # 'INDICACAO_MEDDRA',
    # 'INDICACAO_MEDDRA_SCORE',
    'PK_LLT_LK',   # -> renomear via rename_map
    # 'INDICACAO_MEDDRA_LK',

    # 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL',
    # 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL_SCORE',
    'PK_LLT_MATCH',  # -> renomear via rename_map
    # 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL_MATCH',

    # 'DOSE',
    'DOSE_TIPO_CHAVE',
    'DOSE_TIPO_VALOR',
    'DOSE_VALOR',

    'FREQUENCIA_DOSE',
    'POSOLOGIA',

    # 'DURACAO',
    'DURACAO_TIPO_CHAVE',
    'DURACAO_TIPO_VALOR',
    'DURACAO_VALOR',

    'INICIO_ADMINISTRACAO',
    'FIM_ADMINISTRACAO',

    # 'FORMA_FARMACEUTICA',
    'FORMA_FARMACEUTICA_SCORE',
    'FORMA_FARMACEUTICA_CHAVE',

    'VIA_ADMINISTRACAO',
    'VIA_ADMINISTRACAO_CHAVE',

    'VIA_ADMINISTRACAO_MAE_PAI',
    'VIA_ADMINISTRACAO_MAE_PAI_CHAVE',

    'NUMELO_LOTE',
    # 'ATUALIZADO',
    'HASH_BRONZE',

    'RELACAO_MEDICAMENTO_EVENTO_VALOR',
    'RELACAO_MEDICAMENTO_EVENTO_CHAVE',

    'COMPONENTE_SUSPEITO_VALOR',
    'COMPONENTE_SUSPEITO_CHAVE',

    'ACAO_ADOTADA_VALOR',
    'ACAO_ADOTADA_CHAVE',
]

# Dicionário de renomes: chave = nome original, valor = novo nome
rename_map = {
    'PK_LLT_LK': 'PK_LLT_INDICACAO_MEDDRA',
    'PK_LLT_MATCH': 'PK_LLT_INDICACAO_RELATADA_NOTIFICADOR_INICIAL',
    # adicione outras renomeações aqui, ex:
    # 'NUMELO_LOTE': 'NUMERO_LOTE',
}

# Lista final de colunas já com renomeação aplicada
final_cols = [rename_map.get(col, col) for col in gold_cols]

# Opcional: se quiser aplicar no DataFrame pandas
fat = hist_silver_dedup.rename(columns=rename_map)
fat = fat[final_cols]

In [68]:
fat["HASH_GOLD"] = build_row_hash(fat, final_cols, algo="sha256", sep="|")

In [ ]:
hist_silver_dedup.to_parquet(Path(project_root) / "data/03_gold/fat_medicamentos/fat_medicamentos.parquet", index=False)